In [ ]:
import numpy as np
import pandas as pd
import pickle
import shortuuid
import random
import pandas as pd
import pysam
import logging
import subprocess
import os
import re

from scipy.stats import chi2_contingency
from typing import List
from array import array
from tqdm import tqdm
from itertools import chain
from Bio import SeqIO

def adjust_counts_to_match_total(c: np.ndarray, c_t: np.ndarray) -> np.ndarray:
    """
    Adjusts read counts among cells/spots to match the total counts.

    Parameters:
    - c (np.ndarray): Original counts for the PAS(s).
    - c_t (np.ndarray): Simulated counts for the PAS(s) after allocation.

    Returns:
    - np.ndarray: Adjusted simulated counts with errors corrected.
    """

    t_error = c - c_t.sum(axis=0)
    for i, error in enumerate(t_error):
        if error > 0:
            error_num = int(error)
            offset_base = error_num // c_t.shape[0]
            offset = np.array(
                [offset_base] * (c_t.shape[0] - error_num % c_t.shape[0])
                + [offset_base + 1] * (error_num % c_t.shape[0]),
                dtype=int,
            )
            np.random.shuffle(offset)
            c_t[:, i] += offset
        elif error < 0:
            error_num = abs(int(error))
            while error_num > 0:
                non_zero_indices = np.where(c_t[:, i] > 0)[0]
                subtract_num = min(len(non_zero_indices), error_num)
                if non_zero_indices.size > 0:
                    shuffle_indices = np.random.choice(
                        non_zero_indices, subtract_num, replace=False
                    )
                    c_t[shuffle_indices, i] -= 1
                error_num -= subtract_num
    # Validate the final adjusted counts
    if not np.isclose(c_t.sum(), c.sum()):
        raise ValueError("Adjusted counts do not match the original total counts.")
    return c_t


def allocate_reads_single_pas(count, cell_num=500, dispersion=0.1):
    """
    Allocate reads across a single PAS.

    Parameters:
    - count (int): The count data for the PAS.
    - cell_num (int, optional): The number of cells to simulate. Defaults to 500.
    - dispersion (float, optional): The dispersion parameter for the negative binomial distribution. Defaults to 0.1.

    Returns:
    - np.array: The allocated reads for the PAS.
    """
    phi = np.random.uniform(0.2, 0.8)
    e_t1 = count * phi
    mu_t1 = e_t1 / cell_num
    r_t1 = mu_t1 / dispersion
    p_t1 = r_t1 / (mu_t1 + r_t1)
    e_t1_vec = np.random.negative_binomial(n=r_t1, p=p_t1, size=cell_num)
    c_t1_vec = e_t1_vec.reshape(-1, 1)
    e_t2 = count - e_t1
    mu_t2 = e_t2 / cell_num
    r_t2 = mu_t2 / dispersion
    p_t2 = r_t2 / (mu_t2 + r_t2)
    e_t2_vec = np.random.negative_binomial(n=r_t2, p=p_t2, size=cell_num)
    c_t2_vec = e_t2_vec.reshape(-1, 1)
    c_t1 = np.array(int((count * phi).item()), dtype=int)
    c_t2 = np.array(count - c_t1, dtype=int)
    e_t1_vec = adjust_counts_to_match_total(c_t1, c_t1_vec)
    e_t2_vec = adjust_counts_to_match_total(c_t2, c_t2_vec)
    return e_t1_vec, e_t2_vec


def allocate_reads_multi_pas(
    counts,
    alpha,
    comparison_type,
    cell_num=500,
    dispersion=0.1,
    chi2_p_value=0.05,
    percentage_diff_threshold=0.1,
    swap_prob=0.1,
    pas_num=None,
):
    """
    Allocate reads across multiple PAS based on specified criteria.

    Parameters:
    - counts (np.array): The count data for PAS.
    - alpha (float): Parameter alpha for the beta distribution.
    - comparison_type (str): The type of comparison, 'diff' for differential, 'non_diff' for non-differential.
    - cell_num (int, optional): The number of cells to simulate. Defaults to 500.
    - dispersion (float, optional): The dispersion parameter for the negative binomial distribution. Defaults to 0.1.
    - chi2_p_value (float, optional): P-value threshold for chi-square test. Defaults to 0.05.
    - percentage_diff_threshold (float, optional): Threshold for percentage difference to consider. Defaults to 0.1.
    - pas_num (int, optional): The number of PAS. If not specified, inferred from counts.

    Returns:
    - (np.array, np.array): Tuple of two arrays representing allocated reads for two conditions.
    """
    total_reads = counts.sum()
    phi = np.random.uniform(0.2, 0.8)
    diff_flag = True
    if pas_num is None:
        pas_num = len(counts)
    times = 0
    while diff_flag:
        # Generate delta values using the beta distribution
        delta = np.sort(np.random.beta(alpha, alpha * phi, pas_num))
        # Randomly swap adjacent elements with 10% probability
        for i in range(len(delta) - 1):
            if np.random.rand() < swap_prob:
                delta[i], delta[i + 1] = delta[i + 1], delta[i]

        # Calculate theta for both conditions
        theta_t1 = (counts * delta) / (counts * delta).sum()
        theta_t2 = (counts * (1 - delta)) / (counts * (1 - delta)).sum()

        # Simulate reads allocation for condition 1
        e_t1 = (counts * delta).sum()
        # m_t1_nonzero = int(min(cell_num, max(int(e_t1), 1)))
        mu_t1 = e_t1 / cell_num
        r_t1 = mu_t1 / dispersion
        p_t1 = r_t1 / (mu_t1 + r_t1)
        e_t1_vec = np.random.negative_binomial(n=r_t1, p=p_t1, size=cell_num)
        c_t1_vec = np.array([np.random.multinomial(n, theta_t1) for n in e_t1_vec])

        # Simulate reads allocation for condition 2
        e_t2 = total_reads - e_t1
        # m_t2_nonzero = int(min(cell_num, max(int(e_t2), 1)))
        mu_t2 = e_t2 / cell_num
        r_t2 = mu_t2 / dispersion
        p_t2 = r_t2 / (mu_t2 + r_t2)
        e_t2_vec = np.random.negative_binomial(n=r_t2, p=p_t2, size=cell_num)
        c_t2_vec = np.array([np.random.multinomial(n, theta_t2) for n in e_t2_vec])

        # Adjust for errors in allocation
        c_t1 = (counts * delta).astype(int)
        c_t2 = counts - c_t1
        c_t1_vec = adjust_counts_to_match_total(c_t1, c_t1_vec)
        c_t2_vec = adjust_counts_to_match_total(c_t2, c_t2_vec)
        # Perform chi-square test and evaluate conditions
        chi2_data = [c_t1_vec.sum(axis=0), c_t2_vec.sum(axis=0)]
        chi2, p, dof, expected = chi2_contingency(chi2_data)
        percentage_diff = abs(theta_t1 - theta_t2).sum() / 2

        if comparison_type == "diff":
            if p < chi2_p_value and percentage_diff > percentage_diff_threshold:
                diff_flag = False
        elif comparison_type == "non_diff":
            if p > chi2_p_value and percentage_diff < percentage_diff_threshold:
                diff_flag = False
        else:
            raise ValueError("comparison_type must be 'diff' or 'non_diff'")
        times += 1

    return c_t1_vec, c_t2_vec


"""
# test
c = np.array([20, 200, 25, 30, 40])  # np.random.randint(20, 100000, 5)
alpha = 10
cell_num = 100
dispersion = 0.1
c_t1_vec, c_t2_vec = allocate_reads_multi_pas(c, alpha, "diff", cell_num, dispersion)
print(c_t1_vec.sum(axis=0))
print(c_t2_vec.sum(axis=0))
a, b = allocate_reads_single_pas(np.array([100]), 100)
"""


def reverse_cigar(cigar_str):
    elements = re.findall(r"(\d+)([MIDNSHP=X])", cigar_str)
    return "".join(num + op for num, op in elements[::-1])


def extract_seq_by_cigar(cigar, start_site, seq, soft_clipping_fill="A"):
    sequence_parts = []
    index = start_site
    append_sequence = sequence_parts.append

    cigar_operations = cigar_ops.findall(cigar)
    for num, op in cigar_operations:
        num = int(num)
        if op == "M":
            append_sequence(seq[index : index + num])
            index += num
        elif op == "I":
            append_sequence("N" * num)
        elif op == "D" or op == "N":
            index += num
        elif op == "S":
            append_sequence(soft_clipping_fill * num)
    return "".join(sequence_parts)




def generate_read_list(
    te_list, genome_dict, header, cigar_ops, onsite_flag, logger, console_out=False
):
    read_list = []
    if console_out:
        te_list = tqdm(te_list)
    for te in te_list:
        for pas in te.pas_list:
            current_pas = pas
            read_df = current_pas.read_df.copy()
            sam_flag = 0 if current_pas.strand == "+" else 16
            # reverse cigar
            if current_pas.strand != read_df.strand.iloc[0]:
                read_df.loc[:, "cigar_string"] = read_df.cigar_string.apply(
                    reverse_cigar
                )

            # get read range
            seq_range_numbers = pd.to_numeric(
                read_df.cigar_string.str.extractall(r"(\d+)[MIDN]")[0]
            )
            seq_operations = read_df.cigar_string.str.extractall(r"(\d+)([MIDN])")[1]
            seq_range_numbers.loc[seq_operations == "I"] *= 0
            seq_range = seq_range_numbers.groupby(level=0).sum()

            read_df["start"] = np.where(
                current_pas.strand == "+",
                current_pas.start + read_df["distance_to_pas"] - seq_range,
                current_pas.end - read_df["distance_to_pas"],
            ).astype(int)

            read_df["end"] = np.where(
                current_pas.strand == "+",
                current_pas.start + read_df["distance_to_pas"],
                current_pas.end - read_df["distance_to_pas"] + seq_range,
            ).astype(int)

            current_pas.range_start = read_df["start"].min()
            current_pas.range_end = read_df["end"].max() + 1
            if (
                current_pas.range_start
                < 0 | current_pas.range_end
                > len(genome_dict[current_pas.chr].seq)
            ):
                logger.error(f"start or end out of range: {current_pas}")
                continue
            current_pas.seq = str(
                genome_dict[current_pas.chr].seq[
                    current_pas.range_start : current_pas.range_end
                ]
            )
            read_df.loc[:, "start_site"] = read_df["start"] - current_pas.range_start
            read_df.loc[:, "end_site"] = read_df["end"] - current_pas.range_start

            # extract_seq
            cell_barcode_list = list(current_pas.cell_barcode_generator)
            chr_id = header.references.index(current_pas.chr)
            for i, barcode in enumerate(cell_barcode_list):
                row = read_df.iloc[i]
                if row.is_tail & onsite_flag:
                    soft_clipping_fill = "T" if current_pas.strand == "+" else "A"
                else:
                    soft_clipping_fill = "N"
                read = pysam.AlignedSegment(header=header)
                read_seq = extract_seq_by_cigar(
                    row.cigar_string,
                    row.start_site,
                    current_pas.seq,
                    soft_clipping_fill,
                )
                read.query_sequence = read_seq
                read.query_name = ":".join(
                    [
                        current_pas.chr,
                        str(current_pas.start),
                        current_pas.strand,
                        str(i),
                    ]
                )
                read.flag = sam_flag
                read.reference_id = chr_id
                read.reference_start = row.start
                read.mapping_quality = 60
                read.cigarstring = row.cigar_string
                read.query_qualities = array("B", [30] * len(read_seq))
                read.set_tag("CB", barcode)
                read.set_tag(
                    "UB",
                    "_".join(
                        [
                            barcode,
                            current_pas.chr,
                            str(current_pas.start),
                            current_pas.strand,
                            str(i),
                        ]
                    ),
                )
                read_list.append(read.to_string())
    return read_list

In [ ]:

peak_pickle = "/path/to/apabenchmark_final/data/peaks/Visium_mouse_brain_V19L01-041-C1_peak.pickle"
with open(peak_pickle, "rb") as f:
    peak_list = pickle.load(f)

In [ ]:
# filter and adjust peaks

peak_list_filtered = [x for x in peak_list if (len(x) >= 20)]
retained_peaks = []
drop_peaks = []
for x in tqdm(peak_list_filtered):
    if x.strand.iloc[0] == "+":
        potential_tail = x.cigar_string.str.contains(r"[0-9]{2}S$")
    else:
        potential_tail = x.cigar_string.str.match(r"^[0-9]{2}S")
    potential_pas = x[potential_tail].groupby("distance_to_pas").size()
    potential_pas = potential_pas[potential_pas >= 3]
    if len(potential_pas) > 0:
        pas = potential_pas.idxmax()
        if abs(pas) <= 20:
            x["distance_to_pas"] = x["distance_to_pas"] - pas
            x = x[x["distance_to_pas"] <= 20]
            retained_peaks.append(x)
        else:
            drop_peaks.append(x)
    else: 
        if ((x.distance_to_pas > 20).sum() <= 1):
            x = x[x["distance_to_pas"] <= 20]
            retained_peaks.append(x)
        else:
            drop_peaks.append(x)

retained_peaks = [x for x in retained_peaks if len(x) >= 20]

In [ ]:
class PAS:
    def __init__(
            self,
            chr,
            start,
            end,
            strand,
            read_df
    ):
        self.chr = chr 
        self.start = start 
        self.end = end
        self.strand = strand
        self.read_df = read_df
        self.read_num = read_df.shape[0]
    def __repr__(self):
        return f"{self.chr}:{self.start}-{self.end}({self.strand})"
class TE:
    def __init__(self, te_id, pas_list, apa_type=None):
        self.te_id = te_id
        self.pas_list = pas_list
        self.pas_num = len(pas_list)
        valid_apa_types = ["diff", "non_diff", "single_pas"]
        if len(pas_list) == 1:
            self.apa_type = "single_pas"
        elif apa_type in valid_apa_types:
            self.apa_type = apa_type
        else:
            raise ValueError(f"apa_type must be one of {valid_apa_types}, but got '{apa_type}'")
    def __repr__(self):
        return f"TE {self.te_id} with {self.pas_num} PAS"

def extract_seq_by_cigar(cigar, start_site, seq, cigar_ops, soft_clipping_fill='A'):
    sequence_parts = []
    index = start_site
    append_sequence = sequence_parts.append

    cigar_operations = cigar_ops.findall(cigar)
    for num, op in cigar_operations:
        num = int(num)
        if op == 'M':
            append_sequence(seq[index:index+num])
            index += num
        elif op == 'I':
            append_sequence('N' * num)
        elif op == 'D' or op == 'N':
            index += num
        elif op == 'S':
            append_sequence(soft_clipping_fill * num)
    return ''.join(sequence_parts)

def calculate_peak_range(read_df):
    seq_range_numbers = pd.to_numeric(
        read_df.cigar_string.str.extractall(r"(\d+)[MIDN]")[0]
    )
    seq_operations = read_df.cigar_string.str.extractall(r"(\d+)([MIDN])")[1]
    seq_range_numbers.loc[seq_operations == "I"] *= 0
    seq_range = seq_range_numbers.groupby(level=0).sum()
    start_range = read_df["distance_to_pas"] - seq_range
    end_range = read_df["distance_to_pas"]
    return start_range.min(), end_range.max()

In [ ]:


# input
# genome_dict = SeqIO.to_dict(
#     list(SeqIO.parse("/path/to/GRCm38.p6.genome.fa", "fasta"))
# )
# retained_peaks
pas_path = "/path/to/apabenchmark_final/simulation/pas/mm10_sim_pas_rep1.bed"
pas_sample = pd.read_csv(pas_path, sep="\t")
pas_sample.loc[:,"chr_end"] = pas_sample.apply(lambda x: len(genome_dict[x.chr].seq), axis=1)
pas_sample["end_gap"] = pas_sample["chr_end"] - pas_sample["end"]

sim_bam_path = "/path/to/apabenchmark_final/simulation/mm10_sim_rep1.bam"

min_reads_per_peak = 20
cell_num_per_type = 100
barcode_length = 6
dispersion = 0.1
progress_flag = True

__name__="test"

# set logger
logger = logging.getLogger(__name__ + ".generate_sim_bam")
logger.setLevel(logging.INFO)
logger.basicConfig = False
log_file = f"{sim_bam_path}.log"

file_handler = logging.FileHandler(log_file, mode="w")
file_handler.setLevel(logging.INFO)
formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

# if progress_flag:
#     console_handler = logging.StreamHandler()
#     console_handler.setLevel(logging.INFO)
#     console_handler.setFormatter(formatter)
#     logger.addHandler(console_handler)


# prepare peaks
retained_peaks = [x for x in retained_peaks if len(x) >= min_reads_per_peak]

# process pas_sample
diff_te_list = pas_sample[pas_sample["pas_type"] == "diff_apa"]["exon_id"].unique()
non_diff_te_list = pas_sample[pas_sample["pas_type"] == "non_diff_apa"][
    "exon_id"
].unique()
single_pas_gene_list = pas_sample[pas_sample["pas_type"] == "single_pas"][
    "exon_id"
].unique()
pas_num = pas_sample.shape[0]

# select peak
# keep more peaks than needed to provide enougt choices for TE, prevent lack of eligible peaks
if len(retained_peaks) >= int(pas_num * 1.1):
    retained_peaks = random.sample(retained_peaks, int(pas_num * 1.1))
    random.shuffle(retained_peaks)
else:
    # if not enough peaks, duplicate some peaks
    retained_peaks_additional = [
        random.choice(retained_peaks)
        for _ in range(int(pas_num * 1.1) - len(retained_peaks))
    ]
    retained_peaks += retained_peaks_additional
    random.shuffle(retained_peaks)

logger.info(f"Number of PAS: {pas_num}")
logger.info(f"Number of retained peaks: {len(retained_peaks)}")

t1_cell_barcode_list = [
    shortuuid.ShortUUID().random(length=barcode_length)
    for _ in range(cell_num_per_type)
]
t2_cell_barcode_list = [
    shortuuid.ShortUUID().random(length=barcode_length)
    for _ in range(cell_num_per_type)
]

peak_read_num = [len(x) for x in retained_peaks]
peak_range = [calculate_peak_range(x) for x in retained_peaks]
remaining_set = set(zip(peak_read_num, range(len(peak_read_num)), peak_range))
original_remaining_set = remaining_set.copy()

logger.info(f"Number of remaining peaks: {len(remaining_set)}")
te_group = []

# set alpha=10 to generate differential TEs
# alpha controls the dispersion of the beta distribution
# smaller alpha leads to larger dispersion
alpha = 10
for te in diff_te_list:
    pas_group = pas_sample[pas_sample["exon_id"] == te]
    pas_num = len(pas_group)
    pas_start_gap = pas_group["start"]
    pas_end_gap = pas_group["end_gap"]
    pas_strand = pas_group["strand"].iloc[0]
    if pas_strand == "+":
        pass
    else:
        pas_start_gap, pas_end_gap = pas_end_gap, pas_start_gap
    while True:
        current_group = set(random.sample(list(remaining_set), pas_num))
        # check if the peaks are within the range of the chromsome
        start_positions = pas_start_gap + np.array([x[2][0] for x in current_group])
        end_positions = pas_end_gap - np.array([x[2][1] for x in current_group])
        if (start_positions < 0).any() or (end_positions < 0).any():
            continue

        values_sorted = sorted(current_group, key=lambda x: x[0], reverse=True)
        max_value_in_group = values_sorted[0][0]
        second_max_value_in_group = values_sorted[1][0]
        if max_value_in_group <= 2 * second_max_value_in_group:
            remaining_set -= current_group
            break

        eligible_second_peaks = [
            x
            for x in remaining_set
            if max_value_in_group * 0.5 <= x[0] <= max_value_in_group * 2
        ]
        eligible_second_peaks = [
            x for x in eligible_second_peaks if x not in current_group
        ]

        if not eligible_second_peaks:
            eligible_second_peaks = [
                x
                for x in original_remaining_set
                if max_value_in_group * 0.5 <= x[0] <= max_value_in_group * 2
            ]
            eligible_second_peaks = [
                x for x in eligible_second_peaks if x not in current_group
            ]

        if eligible_second_peaks:
            new_second_peak = random.sample(eligible_second_peaks, 1)
            old_second_max_value_peak = values_sorted[1]
            current_group.discard(old_second_max_value_peak)
            current_group |= set(new_second_peak)

            # Check if the updated peaks are within the range of the chromosome
            start_positions = pas_start_gap + np.array([x[2][0] for x in current_group])
            end_positions = pas_end_gap - np.array([x[2][1] for x in current_group])
            if (start_positions < 0).any() or (end_positions < 0).any():
                continue
            remaining_set -= current_group
            break
        else:
            remaining_set |= current_group
            remaining_set.remove(max(current_group, key=lambda x: x[0]))
        
    current_peak_list = [retained_peaks[x] for _, x, i in current_group]
    pas_list = [
        PAS(x.chr, x.start, x.end, x.strand, current_peak_list[_])
        for _, x in pas_group.sort_values(["chr", "start"]).reset_index().iterrows()
    ]
    pas_counts = np.array([x.read_num for x in pas_list])
    # generate read distribution
    c_t1_vec, c_t2_vec = allocate_reads_multi_pas(
        pas_counts, alpha, "diff", cell_num_per_type, dispersion
    )

    # allocate reads to cell barcode
    for i, pas in enumerate(pas_list):
        pas.cell_barcode_generator = (
            barcode
            for barcode in chain(
                np.repeat(t1_cell_barcode_list, c_t1_vec[:, i]),
                np.repeat(t2_cell_barcode_list, c_t2_vec[:, i]),
            )
        )
    te_group.append(TE(te, pas_list, "diff"))
    te_group[-1].c_t1_vec = c_t1_vec
    te_group[-1].c_t2_vec = c_t2_vec

logger.info(f"Number of remaining peaks: {len(remaining_set)}")

# set alpha=100 to generate non-differential TEs
alpha = 100
for te in non_diff_te_list:
    pas_group = pas_sample[pas_sample["exon_id"] == te]
    pas_num = len(pas_group)
    pas_start_gap = pas_group["start"]
    pas_end_gap = pas_group["end_gap"]
    pas_strand = pas_group["strand"].iloc[0]
    if pas_strand == "+":
        pass
    else:
        pas_start_gap, pas_end_gap = pas_end_gap, pas_start_gap
    while True:
        current_group = set(random.sample(list(remaining_set), pas_num))

        # check if the peaks are within the range of the chromsome
        start_positions = pas_start_gap + np.array([x[2][0] for x in current_group])
        end_positions = pas_end_gap - np.array([x[2][1] for x in current_group])
        if (start_positions < 0).any() or (end_positions < 0).any():
            continue

        values_sorted = sorted(current_group, key=lambda x: x[0], reverse=True)
        max_value_in_group = values_sorted[0][0]
        second_max_value_in_group = values_sorted[1][0]
        if max_value_in_group <= 2 * second_max_value_in_group:
            remaining_set -= current_group
            break

        eligible_second_peaks = [
            x
            for x in remaining_set
            if max_value_in_group * 0.5 <= x[0] <= max_value_in_group * 2
        ]
        eligible_second_peaks = [
            x for x in eligible_second_peaks if x not in current_group
        ]

        if not eligible_second_peaks:
            eligible_second_peaks = [
                x
                for x in original_remaining_set
                if max_value_in_group * 0.5 <= x[0] <= max_value_in_group * 2
            ]
            eligible_second_peaks = [
                x for x in eligible_second_peaks if x not in current_group
            ]

        if eligible_second_peaks:
            new_second_peak = random.sample(eligible_second_peaks, 1)
            old_second_max_value_peak = values_sorted[1]
            current_group.discard(old_second_max_value_peak)
            current_group |= set(new_second_peak)

            # Check if the updated peaks are within the range of the chromosome
            start_positions = pas_start_gap + np.array([x[2][0] for x in current_group])
            end_positions = pas_end_gap - np.array([x[2][1] for x in current_group])
            if (start_positions < 0).any() or (end_positions < 0).any():
                continue

            remaining_set -= current_group
            break
        else:
            remaining_set |= current_group
            remaining_set.remove(max(current_group, key=lambda x: x[0]))

    current_peak_list = [retained_peaks[x] for _, x, i in current_group]
    pas_list = [
        PAS(x.chr, x.start, x.end, x.strand, current_peak_list[_])
        for _, x in pas_group.sort_values(["chr", "start"]).reset_index().iterrows()
    ]
    pas_counts = np.array([x.read_num for x in pas_list])

    # generate read distribution
    c_t1_vec, c_t2_vec = allocate_reads_multi_pas(
        pas_counts, alpha, "non_diff", cell_num_per_type, dispersion
    )

    # allocate reads to cell barcode
    for i, pas in enumerate(pas_list):
        pas.cell_barcode_generator = (
            barcode
            for barcode in chain(
                np.repeat(t1_cell_barcode_list, c_t1_vec[:, i]),
                np.repeat(t2_cell_barcode_list, c_t2_vec[:, i]),
            )
        )
    te_group.append(TE(te, pas_list, "non_diff"))
    te_group[-1].c_t1_vec = c_t1_vec
    te_group[-1].c_t2_vec = c_t2_vec

logger.info(f"Number of remaining peaks: {len(remaining_set)}")

for te in single_pas_gene_list:
    pas_group = pas_sample[pas_sample["exon_id"] == te]
    pas_num = len(pas_group)

    if pas_num != 1:
        raise ValueError("pas_num must be 1 for single_pas_gene_list")

    pas_start_gap = pas_group["start"]
    pas_end_gap = pas_group["end_gap"]
    pas_strand = pas_group["strand"].iloc[0]
    if pas_strand == "+":
        pass
    else:
        pas_start_gap, pas_end_gap = pas_end_gap, pas_start_gap


    while True:
        current_group = set(random.sample(list(remaining_set), 1))
        # check if the peaks are within the range of the chromsome
        start_positions = pas_start_gap + np.array([x[2][0] for x in current_group])
        end_positions = pas_end_gap - np.array([x[2][1] for x in current_group])
        if (start_positions < 0).any() or (end_positions < 0).any():
            continue
        remaining_set -= current_group
        break

    current_peak_list = [retained_peaks[x] for _, x, i in current_group]
    pas_list = [
        PAS(x.chr, x.start, x.end, x.strand, current_peak_list[_])
        for _, x in pas_group.sort_values(["chr", "start"]).reset_index().iterrows()
    ]
    pas_counts = np.array([x.read_num for x in pas_list])

    # generate read distribution
    c_t1_vec, c_t2_vec = allocate_reads_single_pas(
        pas_counts, cell_num_per_type, dispersion
    )

    # allocate reads to cell barcode
    for i, pas in enumerate(pas_list):
        pas.cell_barcode_generator = (
            barcode
            for barcode in chain(
                np.repeat(t1_cell_barcode_list, c_t1_vec[:, i]),
                np.repeat(t2_cell_barcode_list, c_t2_vec[:, i]),
            )
        )

    te_group.append(TE(te, pas_list, "single_pas"))
    te_group[-1].c_t1_vec = c_t1_vec
    te_group[-1].c_t2_vec = c_t2_vec

logger.info(f"Number of remaining peaks: {len(remaining_set)}")
# generate read list
header = pysam.AlignmentHeader.from_dict(
    {
        "HD": {"VN": "1.0"},
        "SQ": [{"LN": len(x.seq), "SN": x.id} for x in genome_dict.values()],
    }
)
cigar_ops = re.compile(r"(\d+)([MIDNSHP=X])")
onsite_flag = True
read_list = generate_read_list(
    te_group, genome_dict, header, cigar_ops, onsite_flag, logger, progress_flag
)

logger.info(f"Number of reads generated: {len(read_list)}")

# write to bam
bam = pysam.AlignmentFile(sim_bam_path, "wb", header=header)
for read in read_list:
    bam.write(pysam.AlignedSegment.fromstring(read, header=bam.header))
bam.close()

logger.info(f"Simulated bam file written to {sim_bam_path}")

result = subprocess.run(["samtools", "sort", "-o", f"{sim_bam_path}.sorted", sim_bam_path])
if result.returncode != 0:
    logger.error(f"Error occurred while running 'samtools sort'. Return code: {result.returncode}")
    raise RuntimeError("Error occurred while running 'samtools sort'")

os.remove(sim_bam_path)
os.rename(f"{sim_bam_path}.sorted", sim_bam_path)
result = subprocess.run(["samtools", "index", sim_bam_path])
if result.returncode != 0:
    logger.error(f"Error occurred while running 'samtools index'. Return code: {result.returncode}")
    raise RuntimeError("Error occurred while running 'samtools index'")


# save expr mtx
pas_name_list = list(
    chain.from_iterable(
        [
            [
                "|".join(
                    [x.te_id, ":".join([y.chr, str(y.start), str(y.end), y.strand])]
                )
                for y in x.pas_list
            ]
            for x in te_group
        ]
    )
)
cell_list = t1_cell_barcode_list + t2_cell_barcode_list
pas_expr_mtx = np.hstack([np.vstack([x.c_t1_vec, x.c_t2_vec]) for x in te_group])
pas_expr_df = pd.DataFrame(pas_expr_mtx, index=cell_list, columns=pas_name_list)
cell_type_df = pd.DataFrame(
    np.array(["T1"] * cell_num_per_type + ["T2"] * cell_num_per_type),
    index=cell_list,
    columns=["cell_type"],
)
pas_expr_df = pd.concat([cell_type_df, pas_expr_df], axis=1)
pas_expr_df.to_csv(f"{sim_bam_path}.expr.tsv", sep="\t", index=True)
logger.info(f"Expression matrix written to {sim_bam_path}.expr.tsv")

file_handler.close() # 关闭handler
logger.removeHandler(file_handler)


In [ ]:
barcode_length=6

# %timeit t1_cell_barcode_list = [shortuuid.ShortUUID().random(length=barcode_length)for _ in range(20000)]
barcode = shortuuid.ShortUUID().random(length=barcode_length)
chr = "chr1"
%timeit hash("_".join([barcode,chr,"1232412","+",str(10),])).to_bytes(8,"big").hex()


In [ ]:
import hashlib
%timeit hashlib.md5("_".join([chr,"1232412","+",str(10),]).encode()).hexdigest()

In [ ]:
hashlib.sha1("_".join([barcode,chr,"1232412","+",str(10),]).encode()).hexdigest()

In [ ]:
hash("_".join([barcode,chr,"1232412","+",str(10),])).to_bytes(8,"big").hex()

In [ ]:
a = hash("_".join([barcode,chr,"1232412","+",str(10),]))
(-a).to_bytes(8,"big")

In [ ]:
import sys
(1 << sys.hash_info.width) - 1

In [ ]:
pas_path = "/path/to/apabenchmark_final/simulation/pas/mm10_sim_pas_rep1.bed"
pas_sample = pd.read_csv(pas_path, sep="\t")
sim_bam_path = "/path/to/apabenchmark_final/simulation/mm10_sim_rep1.bam"

    __name__="test"
def generate_sim_bam(
    genome_fasta_path: str,
    retained_peaks: List[pd.DataFrame],
    pas_sample: pd.DataFrame,
    output_bam_path: str,
    min_reads_per_peak: int = 20,
    cell_num_per_type: int = 100,
    diff_alpha: float = 10,
    non_diff_alpha: float = 100,
    dispersion: float = 0.1,
    barcode_length: int = 6,
    progress_flag: bool = False,
):


    pas_sample.loc[:,"chr_end"] = pas_sample.apply(lambda x: len(genome_dict[x.chr].seq), axis=1)
    pas_sample["end_gap"] = pas_sample["chr_end"] - pas_sample["end"]
    genome_dict = SeqIO.to_dict(
        list(SeqIO.parse("/path/to/GRCm38.p6.genome.fa", "fasta"))
    )

    # set logger
    logger = logging.getLogger(__name__ + ".generate_sim_bam")
    logger.setLevel(logging.INFO)
    logger.basicConfig = False
    log_file = f"{sim_bam_path}.log"

    file_handler = logging.FileHandler(log_file, mode="w")
    file_handler.setLevel(logging.INFO)
    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)

    # prepare peaks
    retained_peaks = [x for x in retained_peaks if len(x) >= min_reads_per_peak]

    # process pas_sample
    diff_te_list = pas_sample[pas_sample["pas_type"] == "diff_apa"]["exon_id"].unique()
    non_diff_te_list = pas_sample[pas_sample["pas_type"] == "non_diff_apa"][
        "exon_id"
    ].unique()
    single_pas_gene_list = pas_sample[pas_sample["pas_type"] == "single_pas"][
        "exon_id"
    ].unique()
    pas_num = pas_sample.shape[0]

    # select peak
    # keep more peaks than needed to provide enougt choices for TE, prevent lack of eligible peaks
    if len(retained_peaks) >= int(pas_num * 1.1):
        retained_peaks = random.sample(retained_peaks, int(pas_num * 1.1))
        random.shuffle(retained_peaks)
    else:
        # if not enough peaks, duplicate some peaks
        retained_peaks_additional = [
            random.choice(retained_peaks)
            for _ in range(int(pas_num * 1.1) - len(retained_peaks))
        ]
        retained_peaks += retained_peaks_additional
        random.shuffle(retained_peaks)

    logger.info(f"Number of PAS: {pas_num}")
    logger.info(f"Number of retained peaks: {len(retained_peaks)}")

    t1_cell_barcode_list = [
        shortuuid.ShortUUID().random(length=barcode_length)
        for _ in range(cell_num_per_type)
    ]
    t2_cell_barcode_list = [
        shortuuid.ShortUUID().random(length=barcode_length)
        for _ in range(cell_num_per_type)
    ]

    peak_read_num = [len(x) for x in retained_peaks]
    peak_range = [calculate_peak_range(x) for x in retained_peaks]
    remaining_set = set(zip(peak_read_num, range(len(peak_read_num)), peak_range))
    original_remaining_set = remaining_set.copy()

    logger.info(f"Number of remaining peaks: {len(remaining_set)}")
    te_group = []

    # set alpha=10 to generate differential TEs
    # alpha controls the dispersion of the beta distribution
    # smaller alpha leads to larger dispersion
    alpha = 10
    for te in diff_te_list:
        pas_group = pas_sample[pas_sample["exon_id"] == te]
        pas_num = len(pas_group)
        pas_start_gap = pas_group["start"]
        pas_end_gap = pas_group["end_gap"]
        pas_strand = pas_group["strand"].iloc[0]
        if pas_strand == "+":
            pass
        else:
            pas_start_gap, pas_end_gap = pas_end_gap, pas_start_gap
        while True:
            current_group = set(random.sample(list(remaining_set), pas_num))
            # check if the peaks are within the range of the chromsome
            start_positions = pas_start_gap + np.array([x[2][0] for x in current_group])
            end_positions = pas_end_gap - np.array([x[2][1] for x in current_group])
            if (start_positions < 0).any() or (end_positions < 0).any():
                continue

            values_sorted = sorted(current_group, key=lambda x: x[0], reverse=True)
            max_value_in_group = values_sorted[0][0]
            second_max_value_in_group = values_sorted[1][0]
            if max_value_in_group <= 2 * second_max_value_in_group:
                remaining_set -= current_group
                break

            eligible_second_peaks = [
                x
                for x in remaining_set
                if max_value_in_group * 0.5 <= x[0] <= max_value_in_group * 2
            ]
            eligible_second_peaks = [
                x for x in eligible_second_peaks if x not in current_group
            ]

            if not eligible_second_peaks:
                eligible_second_peaks = [
                    x
                    for x in original_remaining_set
                    if max_value_in_group * 0.5 <= x[0] <= max_value_in_group * 2
                ]
                eligible_second_peaks = [
                    x for x in eligible_second_peaks if x not in current_group
                ]

            if eligible_second_peaks:
                new_second_peak = random.sample(eligible_second_peaks, 1)
                old_second_max_value_peak = values_sorted[1]
                current_group.discard(old_second_max_value_peak)
                current_group |= set(new_second_peak)

                # Check if the updated peaks are within the range of the chromosome
                start_positions = pas_start_gap + np.array([x[2][0] for x in current_group])
                end_positions = pas_end_gap - np.array([x[2][1] for x in current_group])
                if (start_positions < 0).any() or (end_positions < 0).any():
                    continue
                remaining_set -= current_group
                break
            else:
                remaining_set |= current_group
                remaining_set.remove(max(current_group, key=lambda x: x[0]))
            
        current_peak_list = [retained_peaks[x] for _, x, i in current_group]
        pas_list = [
            PAS(x.chr, x.start, x.end, x.strand, current_peak_list[_])
            for _, x in pas_group.sort_values(["chr", "start"]).reset_index().iterrows()
        ]
        pas_counts = np.array([x.read_num for x in pas_list])
        # generate read distribution
        c_t1_vec, c_t2_vec = allocate_reads_multi_pas(
            pas_counts, alpha, "diff", cell_num_per_type, dispersion
        )

        # allocate reads to cell barcode
        for i, pas in enumerate(pas_list):
            pas.cell_barcode_generator = (
                barcode
                for barcode in chain(
                    np.repeat(t1_cell_barcode_list, c_t1_vec[:, i]),
                    np.repeat(t2_cell_barcode_list, c_t2_vec[:, i]),
                )
            )
        te_group.append(TE(te, pas_list, "diff"))
        te_group[-1].c_t1_vec = c_t1_vec
        te_group[-1].c_t2_vec = c_t2_vec

    logger.info(f"Number of remaining peaks: {len(remaining_set)}")

    # set alpha=100 to generate non-differential TEs
    alpha = 100
    for te in non_diff_te_list:
        pas_group = pas_sample[pas_sample["exon_id"] == te]
        pas_num = len(pas_group)
        pas_start_gap = pas_group["start"]
        pas_end_gap = pas_group["end_gap"]
        pas_strand = pas_group["strand"].iloc[0]
        if pas_strand == "+":
            pass
        else:
            pas_start_gap, pas_end_gap = pas_end_gap, pas_start_gap
        while True:
            current_group = set(random.sample(list(remaining_set), pas_num))

            # check if the peaks are within the range of the chromsome
            start_positions = pas_start_gap + np.array([x[2][0] for x in current_group])
            end_positions = pas_end_gap - np.array([x[2][1] for x in current_group])
            if (start_positions < 0).any() or (end_positions < 0).any():
                continue

            values_sorted = sorted(current_group, key=lambda x: x[0], reverse=True)
            max_value_in_group = values_sorted[0][0]
            second_max_value_in_group = values_sorted[1][0]
            if max_value_in_group <= 2 * second_max_value_in_group:
                remaining_set -= current_group
                break

            eligible_second_peaks = [
                x
                for x in remaining_set
                if max_value_in_group * 0.5 <= x[0] <= max_value_in_group * 2
            ]
            eligible_second_peaks = [
                x for x in eligible_second_peaks if x not in current_group
            ]

            if not eligible_second_peaks:
                eligible_second_peaks = [
                    x
                    for x in original_remaining_set
                    if max_value_in_group * 0.5 <= x[0] <= max_value_in_group * 2
                ]
                eligible_second_peaks = [
                    x for x in eligible_second_peaks if x not in current_group
                ]

            if eligible_second_peaks:
                new_second_peak = random.sample(eligible_second_peaks, 1)
                old_second_max_value_peak = values_sorted[1]
                current_group.discard(old_second_max_value_peak)
                current_group |= set(new_second_peak)

                # Check if the updated peaks are within the range of the chromosome
                start_positions = pas_start_gap + np.array([x[2][0] for x in current_group])
                end_positions = pas_end_gap - np.array([x[2][1] for x in current_group])
                if (start_positions < 0).any() or (end_positions < 0).any():
                    continue

                remaining_set -= current_group
                break
            else:
                remaining_set |= current_group
                remaining_set.remove(max(current_group, key=lambda x: x[0]))

        current_peak_list = [retained_peaks[x] for _, x, i in current_group]
        pas_list = [
            PAS(x.chr, x.start, x.end, x.strand, current_peak_list[_])
            for _, x in pas_group.sort_values(["chr", "start"]).reset_index().iterrows()
        ]
        pas_counts = np.array([x.read_num for x in pas_list])

        # generate read distribution
        c_t1_vec, c_t2_vec = allocate_reads_multi_pas(
            pas_counts, alpha, "non_diff", cell_num_per_type, dispersion
        )

        # allocate reads to cell barcode
        for i, pas in enumerate(pas_list):
            pas.cell_barcode_generator = (
                barcode
                for barcode in chain(
                    np.repeat(t1_cell_barcode_list, c_t1_vec[:, i]),
                    np.repeat(t2_cell_barcode_list, c_t2_vec[:, i]),
                )
            )
        te_group.append(TE(te, pas_list, "non_diff"))
        te_group[-1].c_t1_vec = c_t1_vec
        te_group[-1].c_t2_vec = c_t2_vec

    logger.info(f"Number of remaining peaks: {len(remaining_set)}")

    for te in single_pas_gene_list:
        pas_group = pas_sample[pas_sample["exon_id"] == te]
        pas_num = len(pas_group)

        if pas_num != 1:
            raise ValueError("pas_num must be 1 for single_pas_gene_list")

        pas_start_gap = pas_group["start"]
        pas_end_gap = pas_group["end_gap"]
        pas_strand = pas_group["strand"].iloc[0]
        if pas_strand == "+":
            pass
        else:
            pas_start_gap, pas_end_gap = pas_end_gap, pas_start_gap


        while True:
            current_group = set(random.sample(list(remaining_set), 1))
            # check if the peaks are within the range of the chromsome
            start_positions = pas_start_gap + np.array([x[2][0] for x in current_group])
            end_positions = pas_end_gap - np.array([x[2][1] for x in current_group])
            if (start_positions < 0).any() or (end_positions < 0).any():
                continue
            remaining_set -= current_group
            break

        current_peak_list = [retained_peaks[x] for _, x, i in current_group]
        pas_list = [
            PAS(x.chr, x.start, x.end, x.strand, current_peak_list[_])
            for _, x in pas_group.sort_values(["chr", "start"]).reset_index().iterrows()
        ]
        pas_counts = np.array([x.read_num for x in pas_list])

        # generate read distribution
        c_t1_vec, c_t2_vec = allocate_reads_single_pas(
            pas_counts, cell_num_per_type, dispersion
        )

        # allocate reads to cell barcode
        for i, pas in enumerate(pas_list):
            pas.cell_barcode_generator = (
                barcode
                for barcode in chain(
                    np.repeat(t1_cell_barcode_list, c_t1_vec[:, i]),
                    np.repeat(t2_cell_barcode_list, c_t2_vec[:, i]),
                )
            )

        te_group.append(TE(te, pas_list, "single_pas"))
        te_group[-1].c_t1_vec = c_t1_vec
        te_group[-1].c_t2_vec = c_t2_vec

    logger.info(f"Number of remaining peaks: {len(remaining_set)}")
    # generate read list
    header = pysam.AlignmentHeader.from_dict(
        {
            "HD": {"VN": "1.0"},
            "SQ": [{"LN": len(x.seq), "SN": x.id} for x in genome_dict.values()],
        }
    )
    cigar_ops = re.compile(r"(\d+)([MIDNSHP=X])")
    onsite_flag = True
    read_list = generate_read_list(
        te_group, genome_dict, header, cigar_ops, onsite_flag, logger, progress_flag
    )

    logger.info(f"Number of reads generated: {len(read_list)}")

    # write to bam
    bam = pysam.AlignmentFile(output_bam_path, "wb", header=header)
    for read in read_list:
        bam.write(pysam.AlignedSegment.fromstring(read, header=bam.header))
    bam.close()

    logger.info(f"Simulated bam file written to {output_bam_path}")

    result = subprocess.run(["samtools", "sort", "-o", f"{sim_bam_path}.sorted", output_bam_path])
    if result.returncode != 0:
        logger.error(f"Error occurred while running 'samtools sort'. Return code: {result.returncode}")
        raise RuntimeError("Error occurred while running 'samtools sort'")

    os.remove(output_bam_path)
    os.rename(f"{output_bam_path}.sorted", output_bam_path)
    result = subprocess.run(["samtools", "index", output_bam_path])
    if result.returncode != 0:
        logger.error(f"Error occurred while running 'samtools index'. Return code: {result.returncode}")
        raise RuntimeError("Error occurred while running 'samtools index'")

    # save expr mtx
    pas_name_list = list(
        chain.from_iterable(
            [
                [
                    "|".join(
                        [x.te_id, ":".join([y.chr, str(y.start), str(y.end), y.strand])]
                    )
                    for y in x.pas_list
                ]
                for x in te_group
            ]
        )
    )
    cell_list = t1_cell_barcode_list + t2_cell_barcode_list
    pas_expr_mtx = np.hstack([np.vstack([x.c_t1_vec, x.c_t2_vec]) for x in te_group])
    pas_expr_df = pd.DataFrame(pas_expr_mtx, index=cell_list, columns=pas_name_list)
    cell_type_df = pd.DataFrame(
        np.array(["T1"] * cell_num_per_type + ["T2"] * cell_num_per_type),
        index=cell_list,
        columns=["cell_type"],
    )
    pas_expr_df = pd.concat([cell_type_df, pas_expr_df], axis=1)
    pas_expr_df.to_csv(f"{sim_bam_path}.expr.tsv", sep="\t", index=True)
    logger.info(f"Expression matrix written to {sim_bam_path}.expr.tsv")

    file_handler.close()
    logger.removeHandler(file_handler)

In [ ]:
pas_group["strand"]

In [ ]:
[retained_peaks[x] for _, x, i in current_group]

In [ ]:
pas_group

In [ ]:
def generate_sim_apa_bam(
    genome_fasta_path: str,
    pas_bed_path: str,
    output_bam_path: str,
    min_reads_per_peak: int = 20,
    cell_num_per_type: int = 100,
    barcode_length: int = 6,
    diff_alpha: float = 10,
    non_diff_alpha: float = 100,
    progress_flag: bool = True,
) -> None:



    """
    Generate simulated single-cell APA data by creating a BAM file with reads assigned to different PAS and cell barcodes,
    and a corresponding expression matrix.

    Parameters:
    -----------
    genome_fasta_path : str
        Path to the reference genome FASTA file.
    pas_bed_path : str
        Path to the BED file containing PAS information.
    output_bam_path : str
        Path to the output BAM file to be generated.
    min_reads_per_peak : int, optional (default=20)
        Minimum number of reads required for a peak to be retained.
    cell_num_per_type : int, optional (default=100)
        Number of cells per cell type (T1 and T2).
    barcode_length : int, optional (default=6)
        Length of the cell barcode.
    diff_alpha : float, optional (default=10)
        Alpha value for the beta distribution used to generate differential APA read distributions.
    non_diff_alpha : float, optional (default=100)
        Alpha value for the beta distribution used to generate non-differential APA read distributions.
    progress_flag : bool, optional (default=True)
        Whether to display logging information in the console.

    Returns:
    --------
    None
        The function generates the following output files:
        - Simulated BAM file at the specified output_bam_path.
        - Sorted and indexed BAM file at output_bam_path.
        - Expression matrix TSV file at output_bam_path with '.expr.tsv' extension.
        - Log file at output_bam_path with '.log' extension.

    Raises:
    -------
    TypeError
        If any of the input parameters have an incorrect data type.
    ValueError
        If any of the input parameters have an invalid value.

    Description:
    ------------
    The function performs the following steps:
    1. Validates the input parameters for correct data types and values.
    2. Loads the genome dictionary from the provided FASTA file.
    3. Reads PAS information from the provided BED file.
    4. Filters retained peaks based on the minimum number of reads per peak.
    5. Generates cell barcodes for two cell types (T1 and T2).
    6. Groups PAS into different TE (Transcript End) categories: differential APA, non-differential APA, and single PAS genes.
    7. For each TE group, randomly selects peaks and assigns them to PAS.
    8. Generates read distributions for each PAS based on the TE type and cell type using beta distribution with different alpha values to control dispersion.
    9. Allocates reads to cell barcodes based on the generated read distributions.
    10. Generates a list of simulated reads using the TE groups, genome dictionary, and header information.
    11. Writes the simulated reads to a BAM file.
    12. Sorts and indexes the BAM file using samtools.
    13. Creates an expression matrix with cell barcodes as rows and PAS as columns, containing the read counts for each PAS in each cell.
    14. Saves the expression matrix as a TSV file.

    The function uses the logging module to record information about the progress and output files.
    """
    # Validate input parameter data types and values
    if not isinstance(genome_fasta_path, str):
        raise TypeError("genome_fasta_path must be a string")
    if not isinstance(pas_bed_path, str):
        raise TypeError("pas_bed_path must be a string")
    if not isinstance(output_bam_path, str):
        raise TypeError("output_bam_path must be a string")
    if not isinstance(min_reads_per_peak, int) or min_reads_per_peak <= 0:
        raise ValueError("min_reads_per_peak must be a positive integer")
    if not isinstance(cell_num_per_type, int) or cell_num_per_type <= 0:
        raise ValueError("cell_num_per_type must be a positive integer")
    if not isinstance(barcode_length, int) or barcode_length <= 0:
        raise ValueError("barcode_length must be a positive integer")
    if not isinstance(diff_alpha, float) or diff_alpha <= 0:
        raise ValueError("diff_alpha must be a positive float")
    if not isinstance(non_diff_alpha, float) or non_diff_alpha <= 0:
        raise ValueError("non_diff_alpha must be a positive float")
    if not isinstance(progress_flag, bool):
        raise TypeError("progress_flag must be a boolean")
    

In [ ]:
# subprocess.run(["samtools", "sort", "-o", f"{sim_bam_path}.sorted", sim_bam_path])
# os.remove(sim_bam_path)
# os.rename(f"{sim_bam_path}.sorted", sim_bam_path)
# subprocess.run(["samtools", "index", sim_bam_path])
# subprocess.run(["regtools", "junctions", "extract", "-s", "RF", "-o", f"{sim_bam_path}.junctions.bed", sim_bam_path])
# junctions_df = pd.read_csv(f"{sim_bam_path}.junctions.bed", sep="\t", header=None)
# junctions_df.iloc[:,5] = junctions_df.apply(lambda row: "+" if row[5] == "+" else "-", axis=1)

In [ ]:
pd.read_csv(f"{sim_bam_path}.expr.tsv", sep="\t", index_col=0)

In [ ]:
import re
from array import array
cigar_ops = re.compile(r"(\d+)([MIDNSHP=X])")
onsite_flag = True

read_list = generate_read_list(
    te_group[300:400], genome_dict, header, cigar_ops, onsite_flag, verbose=True
)

In [ ]:
te_group[8].pas_list

In [ ]:
current_pas = te_group[218].pas_list[2]
read_df = current_pas.read_df.copy()
sam_flag = 0 if current_pas.strand == "+" else 16
# reverse cigar
if current_pas.strand != read_df.strand.iloc[0]:
    read_df.loc[:,"cigar_string"] = read_df.cigar_string.apply(reverse_cigar)

# get read range
seq_range_numbers = pd.to_numeric(read_df.cigar_string.str.extractall(r'(\d+)[MIDN]')[0])
seq_operations = read_df.cigar_string.str.extractall(r'(\d+)([MIDN])')[1]
seq_range_numbers.loc[seq_operations == 'D'] *= -1
seq_range_numbers.loc[seq_operations == 'I'] *= 0
seq_range = seq_range_numbers.groupby(level=0).sum()

read_df["start"] = np.where(
    current_pas.strand == "+",
    current_pas.start + read_df["distance_to_pas"] - seq_range,
    current_pas.end - read_df["distance_to_pas"]
    ).astype(int)

read_df["end"] = np.where(
    current_pas.strand == "+",
    current_pas.start + read_df["distance_to_pas"],
    current_pas.end - read_df["distance_to_pas"] + seq_range
    ).astype(int)

current_pas.range_start = read_df["start"].min()
current_pas.range_end = read_df["end"].max()+1
if current_pas.range_start < 0 | current_pas.range_end > len(genome_dict[current_pas.chr].seq):
    raise ValueError("start or end out of range")
current_pas.seq = str(genome_dict[current_pas.chr].seq[current_pas.range_start:current_pas.range_end])
read_df.loc[:,"start_site"] = read_df["start"] - current_pas.range_start
read_df.loc[:,"end_site"] = read_df["end"] - current_pas.range_start

In [ ]:
read_df[read_df.cigar_string == "23M3D91M6S"]

In [ ]:
def extract_seq_by_cigar(cigar, start_site, seq, cigar_ops, soft_clipping_fill='A'):
    sequence_parts = []
    index = start_site
    append_sequence = sequence_parts.append

    cigar_operations = cigar_ops.findall(cigar)
    for num, op in cigar_operations:
        num = int(num)
        if op == 'M':
            append_sequence(seq[index:index+num])
            print(len(seq[index:index+num]) == num, len(seq[index:index+num]), num, seq[index:index+num] == seq[index:index+num+1])
            index += num
        elif op == 'I':
            append_sequence('N' * num)
        elif op == 'N' or op == 'D':
            index += num
        elif op == 'S':
            append_sequence(soft_clipping_fill * num)
    return ''.join(sequence_parts)
a = extract_seq_by_cigar(read_df.loc[97,"cigar_string"], read_df.loc[97,"start_site"], current_pas.seq, cigar_ops, soft_clipping_fill="N")
# read = pysam.AlignedSegment(header=header)
# read_seq = a
# read.query_sequence = read_seq
# read.query_name = ":".join([current_pas.chr, str(current_pas.start), current_pas.strand, str(i)])
# read.flag = sam_flag
# read.reference_id = chr_id
# read.reference_start = row.start_site
# read.mapping_quality = 60
# read.cigarstring = row.cigar_string
# read.query_qualities = array('B',[30] * len(read_seq))
# read.set_tag("CB", barcode)
# read.set_tag("UB", "_".join([barcode, current_pas.chr, str(current_pas.start), current_pas.strand, str(i)]))
# read_list.append(read.to_string())
len(a.strip("N"))

In [ ]:
a = extract_seq_by_cigar("21M1D96M3S", read_df.loc[8,"start_site"], te_group[4].pas_list[1].seq, cigar_ops, soft_clipping_fill="N")
len(a)

In [ ]:
read_df.loc[0,"cigar_string"], read_df.loc[0,"start_site"]

In [ ]:
pas_sample[pas_sample["start"] == 110785530]

In [ ]:
te_group[218].pas_list[2]

In [ ]:
pysam.AlignedSegment.fromstring(read_list[1696], header=bam.header).query_sequence

In [ ]:
len("GGGGAACCCCACTTGGGTTGCACAGGTTTCCGGGGAGGCATTAGAGGCTCCTTTTAAGGATTCTTAGTCTTATTAATAAAAGAGTTTAAGAATGGGCTTAGAAGGAAGCTGGAGGANNN")

In [ ]:
print(read_list[36033])

In [ ]:
from pybedtools import BedTool
if verbose:
    print(f"Find {len(junctions_df)} junctions")

junctions_bed = BedTool.from_dataframe(junctions_df.iloc[:,[0,1,2,3,4,5]])
junctions_df = junctions_bed.to_dataframe()
if verbose:
    print(f"Find {len(junctions_df)} junctions after removing iod regions")


In [ ]:
from itertools import chain
pas_name_list = list(chain.from_iterable(
[[":".join([x.te_id, "|".join([y.chr, str(y.start), str(y.end), y.strand])]) for y in x.pas_list ] for x in te_group]
))
cell_list = t1_cell_barcode_list + t2_cell_barcode_list
pas_expr_mtx = np.hstack([np.vstack([x.c_t1_vec, x.c_t2_vec]) for x in te_group])
pas_expr_df = pd.DataFrame(pas_expr_mtx, index=cell_list, columns=pas_name_list)

In [ ]:
def reverse_cigar(cigar_str):
    elements = re.findall(r'(\d+)([MIDNSHP=X])', cigar_str)
    reversed_elements = elements[::-1]
    return ''.join(num + op for num, op in reversed_elements)

def extract_seq_by_cigar(cigar, start_site, genome, chr_length, cigar_ops, strand='+', soft_clipping_fill='A'):
    sequence_parts = []
    index = start_site-1
    append_sequence = sequence_parts.append

    cigar_operations = cigar_ops.findall(cigar)
    for num, op in cigar_operations:
        num = int(num)
        if index + num > chr_length:
            raise ValueError('CIGAR string goes beyond the length of the chromosome')
        if op == 'M':
            append_sequence(genome[index:index+num])
            index += num
        elif op == 'I':
            append_sequence('N' * num)
        elif op == 'D' or op == 'N':
            index += num
        elif op == 'S':
            append_sequence(soft_clipping_fill * num)
    return ''.join(sequence_parts)

def extract_seq(params):
    read_list = []
    cigar_ops_precompiled = re.compile(r'(\d+)([MIDNSHSPX=])')
    chr_seq = params['chr_seq']
    chr_length = params["chr_length"]
    header = pysam.AlignmentHeader.from_dict(params['header'])
    chrom = params['chrom']
    pas_site = params['pas_site']
    strand = params['strand']
    rev_cigar = params['rev_cigar']
    surrounding_df = params['surrounding_df']
    upstream_limit = params['upstream_limit']
    downstream_limit = params['downstream_limit']
    onsite = params['onsite']
    df = surrounding_df[(surrounding_df["distance_to_pas"] >= -upstream_limit) & (surrounding_df["distance_to_pas"] <= downstream_limit)].copy()
    df["chrom"] = chrom
    df["unmapped_fill"] = "N"
    letters = string.ascii_letters
    CB_string = ''.join(secrets.choice(letters) for i in range(10))
    # remove tail reads if not onsite
    if not onsite:
        df = df[df["is_tail"] == False]
    
    df["onsite"] = df["is_tail"] & onsite

    if rev_cigar:
        df["cigar_string"] = df["cigar_string"].apply(reverse_cigar)
    try:
        if strand == "+":
            df["end"] = df["distance_to_pas"] + pas_site
            df["start"] = df["end"] - df["reference_length"]
            df.loc[(df["onsite"] == True), "unmapped_fill"] = "A"

            for _, row in df.iterrows():
                UB_string = ''.join(secrets.choice(letters) for i in range(5))
                start = row["start"]
                unmapped_fill = row["unmapped_fill"]
                cigar_string = row["cigar_string"]
                # unmapped_length = row["unmatch_length"]

                a = pysam.AlignedSegment(header=header)
                a.query_name = f"{uuid.uuid4()}:{chrom}:{pas_site}:{strand}"
                a.query_sequence = extract_seq_by_cigar(cigar_string, start, chr_seq, chr_length, cigar_ops_precompiled, strand='+', soft_clipping_fill=unmapped_fill)
                #a.query_sequence = str(genome_dict[chrom].seq[start:end]) + unmapped_fill * unmapped_length
                a.flag = 0
                a.reference_id = header.references.index(chrom)
                a.reference_start = start
                a.mapping_quality = np.random.randint(0, 60)
                a.cigarstring = cigar_string
                # if unmapped_length == 0:
                #     a.cigar = [(0, row["match_length"]),]
                # else:
                #     a.cigar = [(0, row["match_length"]), (4, unmapped_length),]
                a.query_qualities = array('B', [np.random.randint(0, 40) for _ in range(len(a.query_sequence))])
                a.set_tag("CB", CB_string, value_type="Z")
                a.set_tag("UB", UB_string, value_type="Z")
                read_list.append(a.to_string())
                # pbar.update(1)

        elif strand == "-":
            df["start"] = pas_site - df["distance_to_pas"]
            # df["end"] = df["start"] + df["match_length"]
            df.loc[(df["onsite"] == True), "unmapped_fill"] = "T"
            for _, row in df.iterrows():
                UB_string = ''.join(secrets.choice(letters) for i in range(5))
                start = row["start"]
                unmapped_fill = row["unmapped_fill"]
                # unmapped_length = row["unmatch_length"]
                unmapped_fill = row["unmapped_fill"]
                cigar_string = row["cigar_string"]

                a = pysam.AlignedSegment(header=header)
                a.query_name = f"{uuid.uuid4()}:{chrom}:{pas_site}:{strand}"
                #a.query_sequence = unmapped_fill * unmapped_length + str(genome_dict[chrom].seq[start:end])
                a.query_sequence = extract_seq_by_cigar(cigar_string, row["start"], chr_seq, chr_length, cigar_ops_precompiled, strand='-', soft_clipping_fill=unmapped_fill)
                a.flag = 16
                a.reference_id = header.references.index(chrom)
                a.reference_start = row["start"]
                a.mapping_quality = np.random.randint(0, 60) 
                a.cigarstring = cigar_string
                # if unmapped_length == 0:
                #     a.cigar = [(0, row["match_length"]),]
                # else:
                #     a.cigar = [(4, unmapped_length), (0, row["match_length"]),]
                a.query_qualities = array('B', [np.random.randint(0, 40) for _ in range(len(a.query_sequence))])
                a.set_tag("CB", CB_string, value_type="Z")
                a.set_tag("UB", UB_string, value_type="Z")
                read_list.append(a.to_string())
                # pbar.update(1)
        else:
            raise ValueError('strand must be either "+" or "-"')
    except Exception as e:
        with open(f"{chrom}:{pas_site}:{strand}:error.log", "w") as f:
            f.write(f"{e}\n")
            f.write(f"{df.to_string()}\n")
        return []
    return read_list